# Maintainers Copilot — Week 7 Colab Notebook

This standalone notebook runs the first full data-and-training path without cloning the repository inside Colab:

1. install dependencies,
2. fetch closed `fastapi/fastapi` issues,
3. build time-aware train/validation/test splits,
4. inspect the split report,
5. fine-tune the first DistilBERT classifier,
6. save the generated data and artifacts if you mount Google Drive.

The notebook mirrors the repository logic so you can work entirely in Colab when local storage is tight.


## Before you start

For the training part, enable a GPU in Colab:

`Runtime -> Change runtime type -> T4 GPU`

The earlier data cells do not need a GPU, but the transformer fine-tuning run should use one.


In [ ]:
!pip -q install transformers datasets scikit-learn wandb accelerate


In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Optional persistence with Google Drive

If you want the generated dataset and trained model to survive after the Colab runtime resets, mount Drive now. If you only want a temporary experiment, skip this cell.


In [ ]:
# Optional: uncomment if you want persistent storage in Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')
#
# from pathlib import Path
# DRIVE_ROOT = Path('/content/drive/MyDrive/maintainers-copilot')
# DRIVE_DATA_DIR = DRIVE_ROOT / 'data'
# DRIVE_ARTIFACT_DIR = DRIVE_ROOT / 'artifacts'
# DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)
# DRIVE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


## 1. Shared constants and file locations


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from collections import Counter
from collections.abc import Iterable, Iterator, Mapping
from dataclasses import asdict, dataclass
from datetime import UTC, datetime
from pathlib import Path
from typing import Any
from urllib.error import HTTPError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

SOURCE_REPO = 'fastapi/fastapi'
ISSUE_STATE = 'closed'
TARGET_LABELS = ('bug', 'feature', 'docs', 'question')
LABEL_TO_TARGET = {
    'bug': 'bug',
    'feature': 'feature',
    'docs': 'docs',
    'question': 'question',
}
LABEL_TO_ID = {
    'bug': 0,
    'feature': 1,
    'docs': 2,
    'question': 3,
}

DATA_DIR = Path('data')
ARTIFACT_ROOT = Path('artifacts/classifier')
DATA_DIR.mkdir(exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)


## 2. JSONL helpers and GitHub issue fetcher


In [ ]:
def read_jsonl(path: Path) -> Iterator[dict[str, Any]]:
    with path.open(encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            payload = json.loads(line)
            if not isinstance(payload, dict):
                raise ValueError(f'Expected an object on line {line_number} of {path}.')
            yield payload


def write_jsonl(path: Path, records: Iterable[Mapping[str, Any]]) -> None:
    with path.open('w', encoding='utf-8') as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False, sort_keys=True))
            handle.write('\n')


def request_json(url: str) -> tuple[Any, dict[str, str]]:
    headers = {
        'Accept': 'application/vnd.github+json',
        'X-GitHub-Api-Version': '2022-11-28',
        'User-Agent': 'maintainers-copilot-colab-notebook',
    }
    token = os.getenv('GITHUB_TOKEN')
    if token:
        headers['Authorization'] = f'Bearer {token}'

    request = Request(url, headers=headers)
    try:
        with urlopen(request) as response:
            return json.load(response), dict(response.headers.items())
    except HTTPError as exc:
        detail = exc.read().decode('utf-8', errors='replace')
        raise RuntimeError(f'GitHub request failed with HTTP {exc.code}: {detail}') from exc


def next_link(link_header: str | None) -> str | None:
    if not link_header:
        return None
    for part in link_header.split(','):
        url_part, *params = part.split(';')
        if any(param.strip() == 'rel=\"next\"' for param in params):
            return url_part.strip()[1:-1]
    return None


def iter_issue_pages(repo: str, state: str, max_pages: int | None = None) -> Iterator[dict[str, Any]]:
    query = urlencode({
        'state': state,
        'per_page': 100,
        'sort': 'created',
        'direction': 'asc',
    })
    url: str | None = f'https://api.github.com/repos/{repo}/issues?{query}'
    pages_seen = 0

    while url is not None and (max_pages is None or pages_seen < max_pages):
        payload, headers = request_json(url)
        pages_seen += 1
        if not isinstance(payload, list):
            raise RuntimeError('GitHub returned an unexpected non-list issue response.')

        for issue in payload:
            if not isinstance(issue, dict):
                raise RuntimeError('GitHub returned an unexpected issue payload.')
            yield issue
        url = next_link(headers.get('Link'))


def fetch_closed_issues(repo: str = SOURCE_REPO, state: str = ISSUE_STATE) -> list[dict[str, Any]]:
    issues: list[dict[str, Any]] = []
    for issue in iter_issue_pages(repo=repo, state=state):
        if 'pull_request' in issue:
            continue
        issue['repo_full_name'] = repo
        issues.append(issue)
    return issues


## 3. Fetch the raw Week 7 dataset

If GitHub rate limits you, create a personal access token and set it in the notebook environment as `GITHUB_TOKEN`, then rerun this cell.


In [ ]:
raw_issues = fetch_closed_issues()
write_jsonl(DATA_DIR / 'raw_issues.jsonl', raw_issues)
print(f'Fetched {len(raw_issues)} closed issues from {SOURCE_REPO}.')


## 4. Normalize records and build temporal splits


In [ ]:
DEFAULT_TEST_RATIO = 0.15
DEFAULT_VAL_RATIO = 0.15


def normalize_records(raw_records: list[dict[str, Any]]) -> tuple[list[dict[str, Any]], Counter[str]]:
    normalized: list[dict[str, Any]] = []
    dropped: Counter[str] = Counter()

    for record in raw_records:
        label_names = {
            label['name'].casefold()
            for label in record.get('labels', [])
            if isinstance(label, dict) and isinstance(label.get('name'), str)
        }
        mapped_targets = {LABEL_TO_TARGET[label] for label in label_names if label in LABEL_TO_TARGET}

        if not mapped_targets:
            dropped['no_supported_target_label'] += 1
            continue
        if len(mapped_targets) > 1:
            dropped['ambiguous_multi_target_label'] += 1
            continue

        number = record.get('number')
        created_at = record.get('created_at')
        if not isinstance(number, int) or not isinstance(created_at, str):
            dropped['missing_required_fields'] += 1
            continue

        repo = record.get('repo_full_name', SOURCE_REPO)
        target = mapped_targets.pop()
        normalized.append({
            'id': f'{repo}#{number}',
            'repo': repo,
            'number': number,
            'title': record.get('title') or '',
            'body': record.get('body') or '',
            'labels': sorted(label_names),
            'target': target,
            'created_at': created_at,
            'closed_at': record.get('closed_at'),
            'url': record.get('html_url'),
        })

    return normalized, dropped


def counts(records: list[dict[str, Any]]) -> Counter[str]:
    return Counter(record['target'] for record in records)


def distribution_from_counts(record_counts: Counter[str], total: int) -> dict[str, float]:
    if total == 0:
        return {}
    return {label: record_counts[label] / total for label in TARGET_LABELS}


def distribution(records: list[dict[str, Any]]) -> dict[str, float]:
    record_counts = counts(records)
    return distribution_from_counts(record_counts, sum(record_counts.values()))


def distribution_distance(candidate: dict[str, float], reference: dict[str, float]) -> float:
    return sum(abs(candidate.get(label, 0.0) - reference.get(label, 0.0)) for label in TARGET_LABELS)


def created_at(record: dict[str, Any]) -> datetime:
    return datetime.fromisoformat(record['created_at'].replace('Z', '+00:00'))


def split_temporally(
    records: list[dict[str, Any]],
    right_ratio: float,
    label: str,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    if not 0 < right_ratio < 1:
        raise ValueError('Split ratio must be between 0 and 1.')

    required_targets = set(TARGET_LABELS)
    overall_distribution = distribution(records)
    left_counts: Counter[str] = Counter()
    right_counts = counts(records)
    best: tuple[float, int] | None = None

    for index, record in enumerate(records[:-1], start=1):
        target = record['target']
        left_counts[target] += 1
        right_counts[target] -= 1

        if not required_targets.issubset(left_counts):
            continue
        if not required_targets.issubset(label for label, count in right_counts.items() if count > 0):
            continue

        right_size = len(records) - index
        size_error = abs((right_size / len(records)) - right_ratio)
        drift = distribution_distance(
            distribution_from_counts(right_counts, right_size),
            overall_distribution,
        )
        score = size_error + drift
        if best is None or score < best[0]:
            best = (score, index)

    if best is None:
        raise ValueError(
            f'Could not create a temporal {label} split containing every target label. '
            'Fetch more issues or inspect label balance first.'
        )

    _, cutoff = best
    return records[:cutoff], records[cutoff:]


def date_range(records: list[dict[str, Any]]) -> dict[str, str]:
    return {
        'oldest_created_at': records[0]['created_at'],
        'newest_created_at': records[-1]['created_at'],
    }


def assert_strict_test_recency(train: list[dict[str, Any]], test: list[dict[str, Any]]) -> None:
    if created_at(train[-1]) >= created_at(test[0]):
        raise AssertionError('The test split is not strictly newer than the training split.')


def build_report(
    splits: dict[str, list[dict[str, Any]]],
    dropped: Counter[str],
    total_raw: int,
) -> dict[str, Any]:
    return {
        'source_repo': SOURCE_REPO,
        'target_labels': list(TARGET_LABELS),
        'total_raw_records': total_raw,
        'total_normalized_records': sum(len(records) for records in splits.values()),
        'dropped_records': dict(sorted(dropped.items())),
        'splits': {
            name: {
                'count': len(records),
                'label_counts': dict(sorted(counts(records).items())),
                'date_range': date_range(records),
            }
            for name, records in splits.items()
        },
    }


def build_dataset(
    raw_records: list[dict[str, Any]],
    test_ratio: float = DEFAULT_TEST_RATIO,
    val_ratio: float = DEFAULT_VAL_RATIO,
) -> tuple[dict[str, list[dict[str, Any]]], dict[str, Any]]:
    normalized, dropped = normalize_records(raw_records)
    ordered = sorted(normalized, key=created_at)

    train_val, test = split_temporally(ordered, right_ratio=test_ratio, label='test')
    relative_val_ratio = val_ratio / (1 - test_ratio)
    train, val = split_temporally(train_val, right_ratio=relative_val_ratio, label='validation')

    splits = {'train': train, 'val': val, 'test': test}
    assert_strict_test_recency(train=train, test=test)
    return splits, build_report(splits=splits, dropped=dropped, total_raw=len(raw_records))


def write_outputs(
    output_dir: Path,
    splits: dict[str, list[dict[str, Any]]],
    report: dict[str, Any],
) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    for name, records in splits.items():
        write_jsonl(output_dir / f'{name}.jsonl', records)
    with (output_dir / 'split_report.json').open('w', encoding='utf-8') as handle:
        json.dump(report, handle, indent=2, sort_keys=True)
        handle.write('\n')


In [ ]:
splits, report = build_dataset(raw_issues)
write_outputs(DATA_DIR, splits, report)
report


## 5. Inspect the split report before training

Check that every split contains all four labels, the test split is newer than train, and no label is vanishingly small.


In [ ]:
for split_name, details in report['splits'].items():
    print(split_name)
    print('  count:', details['count'])
    print('  label_counts:', details['label_counts'])
    print('  date_range:', details['date_range'])
print('dropped_records:', report['dropped_records'])


## 6. Optional: save generated data to Google Drive


In [ ]:
# Optional: uncomment if Drive was mounted earlier.
# for name in ['raw_issues.jsonl', 'train.jsonl', 'val.jsonl', 'test.jsonl', 'split_report.json']:
#     source = DATA_DIR / name
#     target = DRIVE_DATA_DIR / name
#     target.write_bytes(source.read_bytes())
# print('Copied dataset files to', DRIVE_DATA_DIR)


## 7. First DistilBERT fine-tuning experiment


In [ ]:
import numpy as np
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

@dataclass(frozen=True, slots=True)
class TrainingConfig:
    run_name: str
    output_dir: str
    model_name: str = 'distilbert-base-uncased'
    max_length: int = 384
    learning_rate: float = 2e-5
    train_batch_size: int = 16
    eval_batch_size: int = 32
    num_train_epochs: int = 3
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    logging_steps: int = 25
    seed: int = 42
    freeze_encoder_layers: int = 4
    freeze_policy: str = 'freeze lower 4 DistilBERT encoder blocks; train top 2 blocks + head'
    wandb_project: str = 'maintainers-copilot-week7'

@dataclass(frozen=True, slots=True)
class DatasetFingerprint:
    path: str
    sha256: str
    examples: int


def fingerprint_jsonl(path: Path) -> DatasetFingerprint:
    digest = hashlib.sha256()
    examples = 0
    with path.open('rb') as handle:
        for line in handle:
            if line.strip():
                examples += 1
            digest.update(line)
    if examples == 0:
        raise ValueError(f'Dataset split is empty: {path}')
    return DatasetFingerprint(path=str(path), sha256=digest.hexdigest(), examples=examples)


def compose_issue_text(title: str, body: str) -> str:
    return f'{title.strip()}\n\n{body.strip()}'.strip()


def freeze_lower_encoder_layers(model: Any, layer_count: int) -> None:
    encoder_layers = getattr(getattr(model, 'distilbert', None), 'transformer', None)
    layers = getattr(encoder_layers, 'layer', None)
    if layers is None:
        raise ValueError('Freeze policy expects a DistilBERT-style encoder stack.')
    for layer in layers[:layer_count]:
        for parameter in layer.parameters():
            parameter.requires_grad = False


def build_run_manifest(config: TrainingConfig, fingerprints: dict[str, DatasetFingerprint]) -> dict[str, Any]:
    return {
        'created_at': datetime.now(UTC).isoformat(),
        'experiment': 'first_encoder_finetune',
        'config': asdict(config),
        'labels': LABEL_TO_ID,
        'dataset': {name: asdict(fingerprint) for name, fingerprint in fingerprints.items()},
        'freeze_policy': config.freeze_policy,
        'logger': {
            'backend': 'wandb',
            'project': config.wandb_project,
            'run_name': config.run_name,
        },
    }


## 8. Log into Weights & Biases


In [ ]:
import wandb
wandb.login()


## 9. Run training


In [ ]:
RUN_NAME = 'first-distilbert-freeze4'
run_dir = ARTIFACT_ROOT / RUN_NAME
config = TrainingConfig(run_name=RUN_NAME, output_dir=str(run_dir))
fingerprints = {
    'train': fingerprint_jsonl(DATA_DIR / 'train.jsonl'),
    'val': fingerprint_jsonl(DATA_DIR / 'val.jsonl'),
}
manifest = build_run_manifest(config, fingerprints)
run_dir.mkdir(parents=True, exist_ok=True)
(run_dir / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n', encoding='utf-8')

raw_dataset = load_dataset('json', data_files={
    'train': str(DATA_DIR / 'train.jsonl'),
    'validation': str(DATA_DIR / 'val.jsonl'),
})
tokenizer = AutoTokenizer.from_pretrained(config.model_name)


def tokenize(batch: dict[str, list[str]]) -> dict[str, Any]:
    texts = [compose_issue_text(title, body) for title, body in zip(batch['title'], batch['body'], strict=True)]
    encoded = tokenizer(texts, truncation=True, max_length=config.max_length)
    encoded['labels'] = [LABEL_TO_ID[target] for target in batch['target']]
    return encoded


tokenized = raw_dataset.map(tokenize, batched=True)
model = AutoModelForSequenceClassification.from_pretrained(
    config.model_name,
    num_labels=len(LABEL_TO_ID),
    id2label={value: key for key, value in LABEL_TO_ID.items()},
    label2id=LABEL_TO_ID,
)
freeze_lower_encoder_layers(model, config.freeze_encoder_layers)


def compute_metrics(eval_prediction: Any) -> dict[str, float]:
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': float(accuracy_score(labels, predictions)),
        'macro_f1': float(f1_score(labels, predictions, average='macro')),
    }

os.environ.setdefault('WANDB_PROJECT', config.wandb_project)
training_args = TrainingArguments(
    output_dir=str(run_dir / 'checkpoints'),
    learning_rate=config.learning_rate,
    per_device_train_batch_size=config.train_batch_size,
    per_device_eval_batch_size=config.eval_batch_size,
    num_train_epochs=config.num_train_epochs,
    weight_decay=config.weight_decay,
    warmup_ratio=config.warmup_ratio,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=config.logging_steps,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    seed=config.seed,
    report_to=['wandb'],
    run_name=config.run_name,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'],
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()
metrics = trainer.evaluate()
trainer.save_model(str(run_dir / 'model'))
tokenizer.save_pretrained(str(run_dir / 'model'))
(run_dir / 'metrics.json').write_text(json.dumps(metrics, indent=2, sort_keys=True) + '\n', encoding='utf-8')
metrics


## 10. Inspect outputs


In [ ]:
print('Run directory:', run_dir)
for path in sorted(run_dir.rglob('*')):
    if path.is_file():
        print(path)
print('\nrun_manifest.json')
print((run_dir / 'run_manifest.json').read_text())
print('metrics.json')
print((run_dir / 'metrics.json').read_text())


## 11. Optional: save trained artifacts to Google Drive


In [ ]:
# Optional: uncomment if Drive was mounted earlier.
# import shutil
# target_dir = DRIVE_ARTIFACT_DIR / RUN_NAME
# if target_dir.exists():
#     shutil.rmtree(target_dir)
# shutil.copytree(run_dir, target_dir)
# print('Copied artifacts to', target_dir)
